# Unsafe-Net — Multi-Entity YOLOv11s Detector (Colab GPU training)

Stage A of the paper-faithful pipeline (YOLOv4→**YOLOv11s** entity detector).
Trains on the 2,262-frame multi-entity dataset (8 classes; zones excluded from detection).

**Runtime → Change runtime type → GPU (T4/L4/A100) before running.**
Config validated locally on CPU (loss decreases, mAP climbs); MPS was skipped due to a macOS-14 backend bug.
Recipe: SGD lr0=0.01, imgsz 800, 2-stage (freeze→fine-tune), hsv_h=0 to protect vest colour.


## 1. Check GPU


In [ ]:
!nvidia-smi


## 2. Install Ultralytics


In [ ]:
!pip -q install ultralytics
import torch, ultralytics
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 3. Get the dataset
Upload `unsafe-net-detector-dataset.zip` to your Google Drive (root of *My Drive*) once, then run this cell.
It copies the zip to the fast local disk and unzips to `/content/ds`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
ZIP = '/content/drive/MyDrive/unsafe-net-detector-dataset.zip'
assert os.path.exists(ZIP), 'Upload unsafe-net-detector-dataset.zip to My Drive root first!'
!cp "$ZIP" /content/ds.zip
!rm -rf /content/ds && mkdir -p /content/ds && unzip -q /content/ds.zip -d /content/ds
print('train imgs:', len(glob.glob('/content/ds/images/train/*.jpg')))
print('val imgs  :', len(glob.glob('/content/ds/images/val/*.jpg')))
print('train lbls:', len(glob.glob('/content/ds/labels/train/*.txt')))
print('val lbls  :', len(glob.glob('/content/ds/labels/val/*.txt')))


## 4. Write data.yaml (nc=8, absolute Colab paths)


In [ ]:
data_yaml = '''path: /content/ds
train: images/train
val: images/val
nc: 8
names:
  0: Person_Authorized
  1: Person_Unauthorized
  2: Safe_Walkway
  3: Restricted_Zone
  4: Panel_Closed
  5: Panel_Opened
  6: Forklift_Safe_Load
  7: Forklift_Overload
'''
open('/content/ds/data.yaml','w').write(data_yaml)
print(data_yaml)


## 5. Stage 1 — freeze backbone, adapt head (30 epochs)
COCO-pretrained yolo11s.pt auto-downloads. hsv_h=0 keeps green vs red-black vests intact.


In [ ]:
from ultralytics import YOLO
AUG = dict(hsv_h=0.0, hsv_s=0.5, hsv_v=0.4, degrees=0.0, flipud=0.0, fliplr=0.5,
           translate=0.1, scale=0.5, mosaic=0.5, close_mosaic=10, mixup=0.0)
COMMON = dict(data='/content/ds/data.yaml', imgsz=800, batch=16, device=0,
              optimizer='SGD', momentum=0.937, weight_decay=0.0005,
              seed=0, patience=30, plots=True, project='/content/runs', exist_ok=True, **AUG)
# If you hit CUDA OOM on a T4, set batch=8.
m = YOLO('yolo11s.pt')
m.train(name='stage1_frozen', freeze=10, epochs=30, lr0=0.01, **COMMON)


## 6. Stage 2 — unfreeze, fine-tune end-to-end (100 epochs, low LR)


In [ ]:
m2 = YOLO('/content/runs/stage1_frozen/weights/best.pt')
m2.train(name='stage2_finetune', freeze=0, epochs=100, lr0=0.001, **COMMON)
BEST = '/content/runs/stage2_finetune/weights/best.pt'
print('best weights:', BEST)


## 7. Validate + per-class mAP (esp. Person_Authorized vs Unauthorized)


In [ ]:
best = YOLO(BEST)
metrics = best.val(data='/content/ds/data.yaml', imgsz=800, device=0)
names = best.names
print('\nmAP50 =', round(float(metrics.box.map50),4), '| mAP50-95 =', round(float(metrics.box.map),4))
print(f"{'class':22s} {'P':>7} {'R':>7} {'mAP50':>7}")
for i,c in enumerate(metrics.box.ap_class_index):
    print(f'{names[int(c)]:22s} {metrics.box.p[i]:7.3f} {metrics.box.r[i]:7.3f} {metrics.box.ap50[i]:7.3f}')


## 8. Save results back to Drive (weights + curves + confusion matrix)


In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/unsafe-net-detector-out', exist_ok=True)
shutil.copy(BEST, '/content/drive/MyDrive/unsafe-net-detector-out/best.pt')
shutil.make_archive('/content/drive/MyDrive/unsafe-net-detector-out/runs', 'zip', '/content/runs')
print('Saved best.pt + runs.zip to My Drive/unsafe-net-detector-out/')
# Optional direct download:
# from google.colab import files; files.download(BEST)


## Next
Download `best.pt` → put it in the repo, then run `build_relation_tokens` locally (inference on MPS is fine)
to feed YOLO detections + zone geometry into the Transformer (Stage B).
